In [ ]:
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from typing import Literal
from dotenv import load_dotenv
import operator
import re
load_dotenv()

True

In [ ]:
class Task(BaseModel):
    id: int
    title: str
    goal: str = Field(...,description="One sentence describing what the reader should be able to do/understand after this section.")
    bullets: list[str] = Field(...,min_length=3,max_length=5,description="3-5 concrete, non-overlapping subpoints to cover in this section.")
    target_words: int = Field(
        ..., description="Target word count for tihs section (120-450)."
    )
    section_type: Literal["intro", "core", "examples", "checklists", "common_mistakes", "conclusion"] = Field(..., description="use 'common mistakes' exactly one in the plan")
    brief: str = Field(...,description="What to cover")

In [ ]:
class Plan(BaseModel):
    blog_title: str
    audience: str = Field(...,description="Who this blog is for.")
    tone: str = Field(..., description="Writing tone (e.g., practival, crisp, etc).")
    tasks: list[Task]

In [38]:
class Section(TypedDict):
    id: int
    content: str

In [39]:
class State(TypedDict):
    topic: str
    plan: Plan
    sections: Annotated[list[Section], operator.add]
    final: str

In [40]:
llm = ChatOpenAI(model="gpt-4.1-mini")

In [ ]:
def orchestractor(state: State):
    plan = llm.with_structured_output(Plan).invoke(
        [
            SystemMessage(content="You are a senior technical writer and developer advocate. Your job is to produce a "
                    "highly actionable outline for a technical blog post.\n\n"
                    "Hard requirements:\n"
                    "- Create 5–7 sections (tasks) that fit a technical blog.\n"
                    "- Each section must include:\n"
                    "  1) goal (1 sentence: what the reader can do/understand after the section)\n"
                    "  2) 3–5 bullets that are concrete, specific, and non-overlapping\n"
                    "  3) target word count (120–450)\n"
                    "- Include EXACTLY ONE section with section_type='common_mistakes'.\n\n"
                    "Make it technical (not generic):\n"
                    "- Assume the reader is a developer; use correct terminology.\n"
                    "- Prefer design/engineering structure: problem → intuition → approach → implementation → "
                    "trade-offs → testing/observability → conclusion.\n"
                    "- Bullets must be actionable and testable (e.g., 'Show a minimal code snippet for X', "
                    "'Explain why Y fails under Z condition', 'Add a checklist for production readiness').\n"
                    "- Explicitly include at least ONE of the following somewhere in the plan (as bullets):\n"
                    "  * a minimal working example (MWE) or code sketch\n"
                    "  * edge cases / failure modes\n"
                    "  * performance/cost considerations\n"
                    "  * security/privacy considerations (if relevant)\n"
                    "  * debugging tips / observability (logs, metrics, traces)\n"
                    "- Avoid vague bullets like 'Explain X' or 'Discuss Y'. Every bullet should state what "
                    "to build/compare/measure/verify.\n\n"
                    "Ordering guidance:\n"
                    "- Start with a crisp intro and problem framing.\n"
                    "- Build core concepts before advanced details.\n"
                    "- Include one section for common mistakes and how to avoid them.\n"
                    "- End with a practical summary/checklist and next steps.\n\n"
                    "Output must strictly match the Plan schema."),
            HumanMessage(content=f"Topic: {state['topic']}")
        ]
    )
    return {"plan":plan}

In [42]:
def fanout(state: State):
    return [Send("worker",{"task":task, "topic":state["topic"], "plan":state["plan"]}) for task in state["plan"].tasks]

In [ ]:
def worker(payload: dict)->dict:
    task = payload["task"]
    topic = payload["topic"]
    plan = payload["plan"]

    blog_title = plan.blog_title

    section_md = llm.invoke([
        SystemMessage(content="You are a senior technical writer and developer advocate. Write ONE section of a technical blog post in Markdown.\n\n"
        "Hard constraints:\n"
        "- Follow the provided Goal and cover ALL Bullets in order (do not skip or merge bullets).\n"
        "- Stay close to the Target words (±15%).\n"
        "- Output ONLY the section content in Markdown (no blog title H1, no extra commentary).\n\n"
        "Technical quality bar:\n"
        "- Be precise and implementation-oriented (developers should be able to apply it).\n"
        "- Prefer concrete details over abstractions: APIs, data structures, protocols, and exact terms.\n"
        "- When relevant, include at least one of:\n"
        "  * a small code snippet (minimal, correct, and idiomatic)\n"
        "  * a tiny example input/output\n"
        "  * a checklist of steps\n"
        "  * a diagram described in text (e.g., 'Flow: A -> B -> C')\n"
        "- Explain trade-offs briefly (performance, cost, complexity, reliability).\n"
        "- Call out edge cases / failure modes and what to do about them.\n"
        "- If you mention a best practice, add the 'why' in one sentence.\n\n"
        "Markdown style:\n"
        "- Start with a '## <Section Title>' heading.\n"
        "- Use short paragraphs, bullet lists where helpful, and code fences for code.\n"
        "- Avoid fluff. Avoid marketing language.\n"
        "- If you include code, keep it focused on the bullet being addressed.\n"),
        HumanMessage(content=f"Blog:{blog_title}\n"f"Topic:{topic}\n\n"f"Section:{task.title}\n"f"Brief:{task.brief}\n\n""Return only the section content in Markdown")
    ]
    ).content.strip()

    return {"sections":[{"id":task.id,"content":section_md}]}

In [48]:
from pathlib import Path

def reducer(state: State):
    title = state['plan'].blog_title
    sections = sorted(state["sections"],key=lambda section: section["id"])
    body = "\n\n".join(section["content"] for section in sections).strip()

    final_md = f"# {title}\n\n{body}\n"

    filename = title.lower().strip().replace(" ", "_") + ".md"
    output_path = Path(filename)
    output_path.write_text(final_md,encoding="utf-8")
    return {"final": final_md}

In [49]:
g = StateGraph(State)
g.add_node("orchestrator",orchestractor)
g.add_node("worker",worker)
g.add_node("reducer",reducer)

In [50]:
g.add_edge(START,"orchestrator")
g.add_conditional_edges("orchestrator", fanout, ["worker"])
g.add_edge("worker", "reducer")
g.add_edge("reducer",END)

app = g.compile()


In [51]:
out = app.invoke({"topic":"Film Making"})

In [ ]:
llm.invoke("who is pawan kalyan")